# 02 — Silver Transform: SaaS Product & Customer Analytics
Enriches Bronze tables with business-logic derived fields.

| Silver Table | Derived Fields |
|---|---|
| silver_accounts | days_as_customer, churn_risk_flag |
| silver_users | days_since_login, engagement_band |
| silver_events | event_week, feature_category |
| silver_support_tickets | response_band, sla_margin_hrs |

In [ ]:
from pyspark.sql import functions as F


In [ ]:
# ── silver_accounts ──────────────────────────────────────────────────────────
df_acc = spark.table("bronze_accounts")

df_sa = (
    df_acc
    .withColumn("days_as_customer",
        F.datediff(F.coalesce(F.col("churn_date"), F.current_date()), F.col("signup_date")))
    .withColumn("churn_risk_flag",
        ((F.col("health_score") < 50) & (F.col("is_churned") == 0)).cast("int"))
    .withColumn("plan_tier",
        F.when(F.col("plan") == "Enterprise", 4)
         .when(F.col("plan") == "Professional", 3)
         .when(F.col("plan") == "Growth", 2)
         .otherwise(1))
    .drop("_ingested_at")
    .withColumn("_updated_at", F.current_timestamp())
)

df_sa.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("silver_accounts")
print(f"silver_accounts: {df_sa.count():,}")

In [ ]:
# ── silver_users ─────────────────────────────────────────────────────────────
df_usr = spark.table("bronze_users")

df_su = (
    df_usr
    .withColumn("days_since_login",
        F.datediff(F.current_date(), F.col("last_login_date")))
    .withColumn("engagement_band",
        F.when(F.col("logins_last_30_days") >= 20, "Power User")
         .when(F.col("logins_last_30_days") >= 10, "Regular")
         .when(F.col("logins_last_30_days") >= 1,  "Occasional")
         .otherwise("Dormant"))
    .drop("_ingested_at")
    .withColumn("_updated_at", F.current_timestamp())
)

df_su.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("silver_users")
print(f"silver_users: {df_su.count():,}")

In [ ]:
# ── silver_events ────────────────────────────────────────────────────────────
df_ev = spark.table("bronze_events")

df_sev = (
    df_ev
    .withColumn("event_week", F.date_trunc("week", F.col("event_date")))
    .withColumn("feature_category",
        F.when(F.col("feature").isin("Dashboard", "Reports", "Data Export"), "Analytics")
         .when(F.col("feature").isin("API", "Integrations"), "Platform")
         .when(F.col("feature").isin("Automation", "Alerts", "ML Insights"), "Intelligence")
         .otherwise("Administration"))
    .drop("_ingested_at")
    .withColumn("_updated_at", F.current_timestamp())
)

df_sev.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("silver_events")
print(f"silver_events: {df_sev.count():,}")

In [ ]:
# ── silver_support_tickets ───────────────────────────────────────────────────
df_tk = spark.table("bronze_support_tickets")

df_stk = (
    df_tk
    .withColumn("sla_margin_hrs",
        F.round(F.col("sla_target_hrs") - F.col("resolution_hrs"), 2))
    .withColumn("response_band",
        F.when(F.col("resolution_hrs") <= 2,   "Same-Day")
         .when(F.col("resolution_hrs") <= 24,  "Next-Day")
         .when(F.col("resolution_hrs") <= 72,  "This-Week")
         .otherwise("Extended"))
    .drop("_ingested_at")
    .withColumn("_updated_at", F.current_timestamp())
)

df_stk.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("silver_support_tickets")
print(f"silver_support_tickets: {df_stk.count():,}")

In [ ]:
print("\n=== Silver Transform Complete ===")
for tbl in ["silver_accounts", "silver_users", "silver_events", "silver_support_tickets"]:
    cnt = spark.table(tbl).count()
    print(f"  {tbl}: {cnt:,}")